<img src="assets/banner.png" alt="Banner" style="width:100%; border-radius:10px; margin-bottom:20px;">

# 🤖 03 - Model Testing & Inference Engine
**Author:** Mostafa  
**License:** MIT License

This is where the magic happens! I built this notebook as a testing ground for the NLP model I trained. 
It loads the best-performing weights from the training session and allows you to test the model's ability to classify completely new, unseen consumer complaints. Try it out in the interactive cell below!

In [8]:
# --- Import Required Libraries ---
import os
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Automatically select GPU for fast inference if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference Engine running on: {device.type.upper()}")

Inference Engine running on: CUDA


In [9]:
# --- Load the Trained Model & Environment ---
# Point to the directory containing our trained weights and label mappings
CHECKPOINT_DIR = os.path.join("project_outputs", "checkpoints")

try:
    print("Initializing NLP Engine from local checkpoints...")
    
    # 1. Load the Model Architecture and Weights
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
    model = model.to(device)
    model.eval() # Set model to evaluation mode (disables dropout layers for consistent predictions)

    # 2. Load the Tokenizer (Text processor)
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

    # 3. Load Category Mappings (Converting predicted numbers back to text labels)
    with open(os.path.join(CHECKPOINT_DIR, "label_mappings.json"), "r") as f:
        mappings = json.load(f)
        idx2label = {int(k): v for k, v in mappings["idx2label"].items()}
        
    print("✅ System Ready: Model, Tokenizer, and Category Database are online!")
except Exception as e:
    print(f"⚠️ Critical Error loading model. Ensure 'project_outputs' exists. Details: {e}")

Initializing NLP Engine from local checkpoints...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ System Ready: Model, Tokenizer, and Category Database are online!


In [10]:
# --- Define the Inference Pipeline ---
def predict_complaint(text):
    """
    Takes a raw text complaint, tokenizes it, feeds it to the NLP model,
    and returns the predicted category along with a confidence score.
    """
    # 1. Preprocess the input text
    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 2. Forward Pass (No gradients needed for inference)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        
        # 3. Calculate probability scores using Softmax
        probs = F.softmax(logits, dim=1)
        confidence, pred_idx = torch.max(probs, dim=1)
        
        # 4. Map the numerical prediction to the actual text category
        pred_label = idx2label[pred_idx.item()]
        
    # 5. Display the Results neatly
    print("-" * 60)
    print(f"📝 Raw Complaint: '{text}'")
    print(f"🎯 Predicted Category: {pred_label}")
    print(f"📊 Confidence Score: {confidence.item() * 100:.2f}%")
    print("-" * 60)
    
    return pred_label, confidence.item()

In [11]:
# --- 1. Automated Testing Suite ---
# Test Case 1: Credit Card Issue
text_1 = "I have been trying to get my credit card balance corrected for months but the bank is ignoring me."
predict_complaint(text_1)

# Test Case 2: Student Loan Issue
text_2 = "My student loan is accumulating too much interest despite my payments being on time."
predict_complaint(text_2)

# Test Case 3: Debt Collection Issue
text_3 = "There is a false collection account on my credit report from a debt collector that I do not recognize."
predict_complaint(text_3)

------------------------------------------------------------
📝 Raw Complaint: 'I have been trying to get my credit card balance corrected for months but the bank is ignoring me.'
🎯 Predicted Category: credit_reporting
📊 Confidence Score: 50.81%
------------------------------------------------------------
------------------------------------------------------------
📝 Raw Complaint: 'My student loan is accumulating too much interest despite my payments being on time.'
🎯 Predicted Category: mortgages_and_loans
📊 Confidence Score: 93.08%
------------------------------------------------------------
------------------------------------------------------------
📝 Raw Complaint: 'There is a false collection account on my credit report from a debt collector that I do not recognize.'
🎯 Predicted Category: debt_collection
📊 Confidence Score: 70.40%
------------------------------------------------------------


('debt_collection', 0.7040011882781982)

In [12]:
# --- 2. Interactive Testing Environment ---
# Run this cell to manually type in a complaint and test the model live!
user_input = input("Enter a financial complaint to test: ")

if user_input.strip():
    predict_complaint(user_input)
else:
    print("No input provided.")

------------------------------------------------------------
📝 Raw Complaint: 'my credit report'
🎯 Predicted Category: debt_collection
📊 Confidence Score: 67.18%
------------------------------------------------------------
